In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs("outputs/tables", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)

In [2]:
df = pd.read_csv("GEFCom2014_prepared.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["date", "hour"]).reset_index(drop=True)

df["hour"] = df["hour"].astype(int) - 1
df["day_of_week"] = df["day_of_week"].astype(int)
df["month"] = df["month"].astype(int) - 1
df["is_weekend"] = df["is_weekend"].astype(int)

In [3]:
load_features = ['load', 'lag_1', 'lag_24', 'load_roll_mean_24', 'load_roll_std_24']

temp_features = [
    'temp', 'temp_lag_1', 'temp_lag_24',
    'temp_roll_mean_24', 'temp_roll_std_24',
    'CDD', 'HDD'
]

calendar_features = ['hour', 'day_of_week', 'month', 'is_weekend']
target_col = "load"

In [4]:
SEQ_LEN = 24

class DatasetRQ7(Dataset):
    def __init__(self, df):
        self.Xl = df[load_features].values.astype(np.float32)
        self.Xt = df[temp_features].values.astype(np.float32)
        self.Xc = df[calendar_features].values.astype(np.int64)
        self.y = df[target_col].values.astype(np.float32)

    def __len__(self):
        return len(self.Xl) - SEQ_LEN

    def __getitem__(self, idx):
        return (
            torch.tensor(self.Xl[idx:idx+SEQ_LEN]),
            torch.tensor(self.Xt[idx:idx+SEQ_LEN]),
            torch.tensor(self.Xc[idx:idx+SEQ_LEN]),
            torch.tensor(self.y[idx+SEQ_LEN])
        )

dataset = DatasetRQ7(df)

train_size = int(0.8 * len(dataset))
train_ds, test_ds = torch.utils.data.random_split(
    dataset, [train_size, len(dataset) - train_size]
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [5]:
df.describe()


,date,hour,load,temp,day_of_week,month,day,is_weekend,lag_1,lag_24,...,zone_industrial,sum_zones,temp_lag_1,temp_lag_24,temp_roll_mean_24,temp_roll_std_24,CDD,HDD,load_roll_mean_24,load_roll_std_24
count,78696,78696.000000,78696.000000,78696.000000,78696.000000,78696.000000,78696.000000,78696.000000,78696.000000,78696.000000,...,78696.000000,78696.000000,78696.000000,78696.000000,78696.000000,78696.000000,78696.000000,78696.000000,78696.000000,78696.000000
mean,2010-07-06 00:00:00,11.500000,3307.491334,47.631260,2.998170,5.536749,15.756023,0.285453,3307.486569,3307.397403,...,826.778468,3307.491334,47.631320,47.632434,47.631847,5.538399,1.659453,19.028193,3307.450591,499.484333
min,2006-01-09 00:00:00,0.000000,1811.000000,-18.000000,0.000000,0.000000,1.000000,0.000000,1811.000000,1811.000000,...,447.913303,1811.000000,-18.000000,-18.000000,-4.722222,0.386726,0.000000,0.000000,2517.750000,245.775481
25%,2008-04-07 00:00:00,5.750000,2845.000000,33.000000,1.000000,3.000000,8.000000,0.000000,2845.000000,2845.000000,...,709.420888,2845.000000,33.000000,33.000000,33.440972,3.577029,0.000000,2.000000,3098.291667,442.717591
50%,2010-07-06 00:00:00,11.500000,3381.000000,48.666667,3.000000,6.000000,16.000000,0.000000,3381.000000,3381.000000,...,839.425749,3381.000000,48.666667,48.666667,48.986111,5.350328,0.000000,16.333333,3270.833333,489.756956
75%,2012-10-03 00:00:00,17.250000,3708.000000,63.000000,5.000000,9.000000,23.000000,1.000000,3708.000000,3708.000000,...,929.678228,3708.000000,63.000000,63.000000,63.694444,7.290291,0.000000,32.000000,3494.583333,541.814482
max,2014-12-31 00:00:00,23.000000,5506.000000,97.000000,6.000000,11.000000,31.000000,1.000000,5506.000000,5506.000000,...,1456.773056,5506.000000,97.000000,97.000000,87.888889,16.138266,32.000000,83.000000,4664.958333,906.468397
std,NaN,6.922231,579.850915,19.187866,2.000164,3.442014,8.792204,0.451633,579.852148,579.824222,...,149.254267,579.850915,19.187773,19.186085,18.238666,2.522301,4.136584,16.967890,300.123473,85.703217
